<h1 class="text-center">POMDP Planning</h1>



In this practical exercise, we will implement POMDP planning classical algorithms: 
1. track alpha-vectors projection using the Exact Value Iteration. 
3. implement PBVI algorithm 

To that, we will start with the tiger problem see saw during the class

<img align="center" src="./tiger_prob.png" width=500/>


- Select the kernel named "**domaineSA**".


### First, let's define the tiger problem in terms of initial belief state, and transition, observation and reward functions

In [ ]:
import sys
import numpy as np
import random
import matplotlib as mpl
import matplotlib.pyplot as plt
from itertools import permutations

class tiger_pomdp:
    def __init__(self):
        self.states = ["tiger-left", "tiger-right"]
        self.actions = ["listen", "open-left", "open-right"]
        self.observations = ["obs-tiger-left", "obs-tiger-right"]
        self.T = np.zeros( (len(self.actions),len(self.states),len(self.states)) )                          
        self.O = np.zeros( (len(self.actions),len(self.observations),len(self.states)) )
        self.R = np.zeros( (len(self.actions),len(self.states)) )
        self.set_transitionFunction()
        self.set_observationFunction()
        self.set_rewardFunction()
        self.set_initialBelief()
        self.gamma = 0.95
        
    def set_transitionFunction(self):
        for a in range(len(self.actions)):
            if self.actions[a] == "listen" :
                self.T[a,:,:] = np.identity(len(self.states))
            else :
                self.T[a,:,:] = np.array([[0.5, 0.5], [0.5, 0.5]])
        return
                
    def set_observationFunction(self):
        for a in range(len(self.actions)):
            if a==0: #listen
                self.O[a,:,:] = np.array([[0.85, 0.15], [0.15, 0.85]])
            else:
                self.O[a,:,:] = np.array([[0.5, 0.5], [0.5, 0.5]])
            
        return
    
    def set_rewardFunction(self):
        self.R = np.array([[-1.0, -1.0], [-100.0, 10.0], [10.0, -100.0]])
        return
        
    def set_initialBelief(self):
        self.init_b = np.ones(len(self.states))*(1.0/len(self.states))
        

tiger = tiger_pomdp()
#print("Transition Function : \n", tiger.T)
#print("\nObservation Function : \n",tiger.O)
#print("\nReward Function : \n",tiger.R)
#print("\nInitial belief state \n", tiger.init_b)

### Now that the POMDP problem is declared, we will define alphavectors data structure that will be used to represent the value function and the policy

we also define some nice methods to initialize the value function and plot it !

In [ ]:
class alphavec:
    def __init__(self):
        self.vectors = np.array([])
        self.actions = []
        
    def set_size_vector_list(self, sz):
        # sz is a tuple (rows,cols)
        self.vectors = np.zeros(sz)
        #self.action = [-1]*sz[0]
        
    def initialize_from(self, R):
        self.vectors = R.copy()
        for a in range(len(R)):
            self.actions.append(a)
    
    def is_present(self, vector, action):
        for v in range(len(self.vectors)):
                if np.array_equal(self.vectors[v],vector) and self.actions[v] == action :
                    return True
        return False
    
    def plot(self):
        actions, counts = np.unique(self.actions, return_counts=True)
        colors = ['r', 'g', 'b']
        bel = np.linspace(0.0, 1.0, num=50)
        print("There is ", len(self.vectors), " vectors")
        for v in range(len(self.vectors)):
            val = bel*(self.vectors[v,0]-self.vectors[v,1]) + self.vectors[v,1]
            plt.plot(bel, val, colors[self.actions[v]])
            plt.ylabel('Value(s0)')
            plt.xlabel('belief(s0)')
            plt.xlim(0.0,1.0)
            plt.ylim(-100,50)
        plt.show()
        

### The code here bellow computes all the projections used in the Exact Value Interation algorithm.
### This code is then exploited to perform few iterations. 
The goal here is to allow you to visualize the exponential number of vectors being generated

In [ ]:
## This function computes all possible projections (Vn+1 = LVn)
################################################################################
def compute_all_projections(alphavectors, R, T, O, gamma=0.95, verbose = False):
    vectors = alphavectors.vectors
    vectors_actions = alphavectors.actions
    Gamma_a = R.copy()
    nb_actions, nb_observations, nb_states = len(T), len(O[0]), np.size(T,1)
    nb_vectors = len(vectors)
    Gamma_ao= np.zeros( (nb_actions, nb_observations, nb_vectors, nb_states) )
    # creates the projections Gamma_ao
    for a in range(nb_actions): # fix action        
        for o in range(nb_observations): # fix observation
            for alpha in range(len(vectors)): # fix a vector
                Gamma_ao[a,o, alpha, :] = gamma*np.sum(np.multiply(O[a,o,:],
                                                                   np.multiply(T[a,:,:],vectors[alpha])),axis=0)   
    # proceeding with the cross-sum
    nb_proj = nb_actions*(nb_vectors**nb_observations) # number exact of projections
    new_alphavectors = alphavec()
    new_alphavectors.set_size_vector_list((nb_proj,nb_states))
    index = 0
    for a in range(nb_actions):
        for alpha_1 in range(nb_vectors):
            for alpha_2 in range(nb_vectors):
                new_alphavectors.vectors[index,:] = Gamma_a[a] + Gamma_ao[a,0,alpha_1] + Gamma_ao[a,1,alpha_2]
                new_alphavectors.actions.append(a)
                index = index+1
    # prints the new vectors obtained if verbose TRUE
    if verbose :
        for i in range(len(new_alphavectors.vectors)):
            print(new_alphavectors.actions[i], " ", new_alphavectors.vectors[i])
            
    return new_alphavectors


## This piece of code performes some iterations of the EXACT value iteration algorithm
################################################################################
## first it initializes the value function using the rewards
alphavectors = alphavec()
alphavectors.initialize_from(tiger.R)

print("Initial value function")
alphavectors.plot()
## than it launches (only) 2 iterations
N = 2
for i in range(N):
        new_alphavectors = compute_all_projections(alphavectors, tiger.R, tiger.T, tiger.O, verbose = False)
        print("Value function at iteration : ", i+1)
        new_alphavectors.plot()
        alphavectors = new_alphavectors

## Note that to stop such a value iteration algorithm, it would be
## necessary to verify for all beliefs in the Belief Space that
## |V_n(b) - V_n+1(b)| < epsilon
## What is not (at all) easy to obtain


### As saw in course, there is no need to go further to understand that it exploses !!
* Some algorithms in literature where interested in pruning dominated alpha-vectors (i.e. the ones that are dominated for any other over the entire belief space)
* Others, propose to approximate the value function for a finite set of belief states, called the reacheable belief states.

### How to determine the best set of reacheable belief states ?? 
* In the following, we implemented a function to generate belief states following the dynamics of the pomdp model using sampling methods
* For the tiger problem, you can check that there are few reacheable belief states !!


In [ ]:
def compute_reacheable_beliefs(pomdp_prob, nbeliefs = 100, tolerance = 1e-2):
    beliefset = list()
    beliefset.append(pomdp_prob.init_b)
    counter = 0
    while len(beliefset) < nbeliefs:
        ## sample an action
        a = random.choice(pomdp_prob.actions)
        a = pomdp_prob.actions.index(a)
        ## sample an observation
        o = random.choice(pomdp_prob.observations)
        o = pomdp_prob.observations.index(o)
        ## sample a belief from the beliefset
        bel0 = random.choice(beliefset)
        ## update the belief using the Bayes' rule
        bel1 = np.inner(pomdp_prob.T[a,:,:],bel0)
        bel1 = np.multiply(pomdp_prob.O[a,o,:],bel1)
        bel1 = (bel1/np.sum(bel1))
        ## add the new belief into the set if it is not yet there (given a tolerance)
        present = False
        for belief in beliefset:
            if np.sum(np.isclose(belief, bel1, rtol=tolerance, atol=tolerance)) == len(belief):
                present = True
        if not present:
            beliefset.append(bel1.copy())
        counter = counter + 1
        ### if we sample several times without increase the set we stop
        if counter == 4*nbeliefs:
            break

    return beliefset

beliefset = compute_reacheable_beliefs(tiger, nbeliefs=50, tolerance = 1e-2)
print("There are only ", len(beliefset), " reacheable belief states !!\n")
print(beliefset)


### Point-based algorithms make use of this trick to approximate the value function
* to that they (try) to expand the set of reacheble belief states at each iteraction
* then, they perform an approximate value iteraction using the backup operation

This **backup** operation consists in:
1. generating a first set of vectors with Gamma_a
2. then for each action, it accounts with the future by summing up observations but...
3. for each observation it keeps only the Gamma_ao projection that maximize the value for a given belief state
4. then it chooses the vector (among the actions) that maximizes the value for this given belief state

**PBVI** implements such operations.
<img align="center" src="./pbvi.png" width=300/>

In the following, you may **complete** the code of the **solve method** to implement the PBVI algorithm based on the pseudo-code above
* You may complete the code where you find *#HERE*

In [ ]:
class PBVI:
    def __init__(self, pomdp_prob, epsilon=0.01, nbeliefs=100):
        self.pomdp = pomdp_prob
        self.epsilon = 0.01
        self.beliefset = []
        self.nbeliefs = nbeliefs
        self.alphavectors = alphavec()
        
    def initialize_valuefunction(self):
        self.alphavectors.initialize_from(self.pomdp.R)
        return
        
    def expand_reacheable_beliefset(self, n=5):
        ## extends the belief set n times
        size = len(self.beliefset)
        counter = 0
        while len(self.beliefset) < (size + n):
            ## sample an action
            a = random.choice(self.pomdp.actions)
            a = self.pomdp.actions.index(a)
            ## sample an observation
            o = random.choice(self.pomdp.observations)
            o = self.pomdp.observations.index(o)
            ## sample a belief from the beliefset
            bel0 = random.choice(self.beliefset)
            ## update the belief
            bel1 = np.inner(self.pomdp.T[a,:,:],bel0)
            bel1 = np.multiply(self.pomdp.O[a,o,:],bel1)
            bel1 = (bel1/np.sum(bel1))
            ## add the new belief into the set if it is not yet there (given a tolerance)
            present = False
            for belief in self.beliefset:
                if np.sum(np.isclose(belief, bel1, rtol=1e-02, atol=1e-02)) == len(belief):
                    present = True
            if not present:
                self.beliefset.append(bel1)
            counter = counter + 1
            ## if belief set does not expands after ~ 4 x n times ... 
            ## we stop, because it is likely no more new beliefs will appear
            if counter == 4*n:
                break
        return
    
    def compute_ao_projections(self, alphavectors): ## computes Gamma_ao
        vectors = alphavectors.vectors
        vectors_actions = alphavectors.actions
        ## initialize structures
        nb_actions, nb_observations, nb_states = len(self.pomdp.actions), len(self.pomdp.observations), len(self.pomdp.states)
        nb_vectors = len(vectors)
        Gamma_ao= np.zeros( (nb_actions, nb_observations, nb_vectors, nb_states) )
        # fix action
        for a in range(nb_actions):
            # fix observation        
            for o in range(nb_observations):
                # for each vector
                for alpha in range(len(vectors)):
                    Gamma_ao[a, o, alpha, :] = np.sum(np.multiply(self.pomdp.O[a,o,:], 
                                                                       np.multiply(self.pomdp.T[a,:,:], 
                                                                                   vectors[alpha])),axis=1)
        return Gamma_ao
    
    ## performs a backup operation
    def compute_backup(self, belief, Gamma_ao): 
        Gamma_a = self.pomdp.R.copy()
        vectors = np.zeros( (len(self.pomdp.actions),len(self.pomdp.states)) )
        # construct the gamma_ab candidates
        for a in range(len(self.pomdp.actions)):
            vectors[a,:] = Gamma_a[a,:].copy()
            for o in range(len(self.pomdp.observations)):
                # get the best alphavec for this (belief,a,o) tuple
                gamma_ao_index = np.argmax(np.inner(Gamma_ao[a,o,:], belief), axis=0)
                vectors[a,:] = vectors[a,:] + self.pomdp.gamma*Gamma_ao[a,o,gamma_ao_index,:].copy()
            
        ## get the index of the alpha vector which the max value among actions for this belief
        act_index = np.argmax(np.inner(vectors, belief), axis=0)
        return (vectors[act_index,:].copy(), act_index)
    
    ## removes repeated vectors
    def removing_duplicatedvectors(self, alphavectors): 
        new_alphavectors = alphavec()
        new_alphavectors.set_size_vector_list((0,len(alphavectors.vectors[0,:])) )
        for v in range(len(alphavectors.vectors)):
            if not new_alphavectors.is_present(alphavectors.vectors[v], alphavectors.actions[v]):
                new_alphavectors.vectors = np.append(new_alphavectors.vectors, 
                                                     [alphavectors.vectors[v].copy()], axis=0)
                new_alphavectors.actions.append(alphavectors.actions[v])
        return new_alphavectors
    
    # checks the convergence of the value function
    def has_converged(self, alphavectors_before, alphavectors, epsilon = 0.0005): 
        converged = []
        vectors_bef = alphavectors_before.vectors
        vectors_now = alphavectors.vectors
        for belief in self.beliefset:
            val_bef = np.max(np.inner(belief,vectors_bef), axis=0)
            val = np.max(np.inner(belief,vectors_now), axis=0)
            if np.abs(val-val_bef)<= epsilon:
                converged.append(True)
        return np.sum(converged) == len(self.beliefset)
    
    ## Implements the PBVI algorithm
    def solve(self, max_iter = 50):
        ## initializes beliefset and value function representation
        self.initialize_valuefunction()
        self.alphavectors.plot()
        n = 0
        self.beliefset.append(self.pomdp.init_b)
        check_solved = False

        while (n < max_iter):
            print("Iteration ", n+1)
            ## First expand the beliestate set
            self.expand_reacheable_beliefset(n=50)
            print("Belief set size:", len(self.beliefset))
            ## new value function
            new_alphavectors = alphavec()
            new_alphavectors.set_size_vector_list( (len(self.beliefset),len(self.pomdp.states)) )
            index = 0
            ## compute Gamma_ao projections
            print("Performing projections ")
            #### Use the compute_ao_projections method and the current set of self.alphavectors
            Gamma_ao = #HERE 
            ## performs backup operator for each belief
            print("Performing backups ")
            for belief in self.beliefset :
                (vector, action) = # HERE 
                new_alphavectors.vectors[index,:] = vector
                new_alphavectors.actions.append(action)
                #print("vector :", new_alphavectors.vectors[index,:], " action : ", new_alphavectors.actions[-1])
                index = index+1
            # remove duplicates
            print("Removing duplicates ")
            new_alphavectors = #HERE 
            print("Checking convergence ")
            check_solved = #HERE 
            self.alphavectors = new_alphavectors
            self.alphavectors.plot()
            n=n+1            
            if check_solved:
                break           
            
        return
    
    def get_action(self, belief):
        act_index = np.argmax(np.inner(self.alphavectors.vectors,belief), axis=0)
        return self.alphavectors.actions[act_index]    
    
    
### launches PBVI on the tiger problem
pbvi = PBVI(pomdp_prob = tiger)
pbvi.solve(max_iter = 200)

### Now, let's see the value function obtained

In [ ]:
for i in range(len(pbvi.alphavectors.vectors)):
    print("Action : ", pbvi.alphavectors.actions[i], "Vector : ", pbvi.alphavectors.vectors[i])

### What does such a resulting policy ??

To that we will simulate the model by sampling episodes and checking step-by-step which action is chosen given a belief state
* to that you can call the pbvi.get_action method to play an action at each step
* complete the code bellow where you find *#HERE*

In [ ]:
### playing with the policy obtained
episodes = 100
steps = 30
cummulated_rews=[]
discounted_cummulated_rews=[]
for episode in range(episodes):
    # starts a new episode
    belief = tiger.init_b
    hidden_state = np.random.choice(tiger.states, p=belief)
    hs = tiger.states.index(hidden_state)
    cum_rew , disc_cum_rew = 0.0, 0.0
    print("Initial belief :", belief)
    for step in range(steps):
        print(" True hidden state : ", tiger.states[hs])
        # geting an action from policy : use pbvi.get_action() method
        a = #HERE
        # sampling the next hidden_state
        phs = hs
        hs = np.random.choice(tiger.states, p=tiger.T[a,phs,:])
        hs = tiger.states.index(hs)
        # sampling an observation
        o = np.random.choice(tiger.observations, p=tiger.O[a,:,hs])
        o = tiger.observations.index(o)
        # updating belief given the action a (from the policy) and the observation o received
        new_belief = np.inner(tiger.T[a,:,:], belief)
        new_belief = np.multiply(tiger.O[a,o,:],new_belief)
        new_belief = (new_belief/np.sum(new_belief))
        print("Action : ", tiger.actions[a])
        print(" Obs : ", tiger.observations[o])
        # receiving rewards and computing expected (discounted) return
        cum_rew+=tiger.R[a,phs]
        disc_cum_rew+=tiger.gamma*tiger.R[a,phs]
        print(" Reward : ", tiger.R[a,phs])        
        print("New belief : ", new_belief)
        belief = new_belief.copy()
        
    cummulated_rews.append(cum_rew)
    discounted_cummulated_rews.append(disc_cum_rew)

## let's average the rewards achieved on these simulations
print("Average cummulated rewards : ", np.mean(cummulated_rews))
print("Average discounted cummulated rewards : ", np.mean(discounted_cummulated_rews))
    